#  Introducting MCP

MCP is a universal plug that allows AI to acess ecternal servers or services like notion gmail etc

## MCP CLients
MCP client serves as the commiunication bridge between the server and MCP servers  acess point to all the tools that an mCP server provides, handling the message and exchange protocol details.

### Transport Agnostic Communication
Key strength of MCP - client and server can communicate over different protocols depending on the setup

The most common setup runs both the MCP client and server on the same machine, communicating through standard input/output. But you can also connect them over:

+ HTTP
+ WebSockets
+ Various other network protocols

## How It All Works Together
Here's a complete example showing how a user query flows through the entire system - from your server, through the MCP client, to external services like GitHub, and back to Claude.

Let's say a user asks "What repositories do I have?" Here's the step-by-step flow:

User Query: The user submits their question to your server
Tool Discovery: Your server needs to know what tools are available to send to Claude
List Tools Exchange: Your server asks the MCP client for available tools
MCP Communication: The MCP client sends a ListToolsRequest to the MCP server and receives a ListToolsResult
Claude Request: Your server sends the user's query plus the available tools to Claude
Tool Use Decision: Claude decides it needs to call a tool to answer the question
Tool Execution Request: Your server asks the MCP client to run the tool Claude specified
External API Call: The MCP client sends a CallToolRequest to the MCP server, which makes the actual GitHub API call
Results Flow Back: GitHub responds with repository data, which flows back through the MCP server as a CallToolResult
Tool Result to Claude: Your server sends the tool results back to Claude
Final Response: Claude formulates a final answer using the repository data
User Gets Answer: Your server delivers Claude's response back to the user



## Defining Tools with MCP

creating a document management MCP server with two core tools , one to read documents and another to update them 


In [8]:
%pip uninstall -y mcp mcp-types
%pip install "mcp[cli]<2"

Found existing installation: mcp 1.30.0
Uninstalling mcp-1.30.0:
  Successfully uninstalled mcp-1.30.0
Found existing installation: mcp-types 2.3.0
Uninstalling mcp-types-2.3.0:
  Successfully uninstalled mcp-types-2.3.0
Note: you may need to restart the kernel to use updated packages.
  Using cached mcp-1.30.0-py3-none-any.whl.metadata (9.3 kB)
Using cached mcp-1.30.0-py3-none-any.whl (234 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Setting up the MCP server
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DocumentMCP", log_level="ERROR")

In [2]:
docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project's budget and expenditures",
    "outlook.pdf": "This document presents the projected future performance of the system",
    "plan.md": "The plan outlines the steps for the project's implementation.",
    "spec.txt": "These specifications define the technical requirements for the equipment"
}

In [5]:
# creating a document reader tool
from pydantic import Field

@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string."
)
def read_document(
    doc_id: str = Field(description="Id of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    
    return docs[doc_id]

In [6]:
# building a document ediitor tool

# simple find and replace operation on documents

@mcp.tool(
    name = 'edit_document',
    description = 'edit a document by replacing a string in the document content with a new string'
    )
def edit_document(
    doc_id : str = Field(description='id of the document that will be edited'),
    old_str : str = Field(description = 'The text to replace.Must match exacrly , including whitespaces'),
    new_str : str = Field(description= 'The new text to insert to place of the old text')
) :
    if doc_id not in docs:
        raise ValueError(f'doc with id{doc_id} not found')
    docs[doc_id] = docs[doc_id].replace(old_str,new_str)    


## The Server inspector

to test the functionality without connectung to a full application python MCP SDK includes a built in browser based inspector

## Implementing a client
MCP Server working , build the client side - client is what allows our application code to communicate with the MCP server and acess its functionality

### understanding the client architecture
The MCP client consists of two main components:

+ MCP Client - A custom class we create to make using the session easier
+ Client Session - The actual connection to the server (part of the MCP Python SDK)


